# Avaliação: Deep Reinforcement Learning

A proposta do trabalho é resolver um problema diferente do que foi visto em sala de aula.

Usando a biblioteca `gymnasium`, escolha outro ambiente para investigar e tentar resolver usando o _Deep Q-learning_.

Seguem algumas sugestões, em ordem de dificuldade:
- [MountainCar](https://gymnasium.farama.org/environments/classic_control/mountain_car/) (`env = "MountainCar-v0"`)
- [Pendulum](https://gymnasium.farama.org/environments/classic_control/pendulum/) (`env = "Pendulum-v1"`)
- [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) (`env = "LunarLander-v3"`)

## Setup

### Imports

In [ ]:
import os
import gymnasium as gym
import torch

from stable_baselines3 import DQN
from stable_baselines3.common.logger import configure
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

### Configurar GPU para Treinamento do Modelo

GPU:
- MPS (Apple's _Metal Performance Shaders_ backend)

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Selected device:", device)

### Definit Ambiente e Política

In [ ]:
env = gym.make(
    id="MountainCar-v0",
    render_mode="rgb_array"
)
POLICY = "MlpPolicy"

## Funções

Criar Modelo

In [ ]:
def create_model(env, policy, device) -> DQN:
    """
    Initialize the DQN model with core hyperparameters.

    :return: A pre-configured instance of a Deep Q-learning model.
    :rtype: DQN
    """

    return DQN(
        policy=policy,
        env=env,
        device=device,                  # GPU (Apple Silicon MPS)
        learning_rate=1e-4,             # Tamanho do "step" do otimizador (gradiente)
        gamma=0.99,                     # Fator de desconto (recompensas futuras)
        buffer_size=50000,              # Tamanho do buffer de replay
        learning_starts=1000,           # Delay antes de mandar atualizações
        batch_size=128,                 # Tamanho do batch do otimizador (gradiente)
        train_freq=4,                   # Treina a cada n passos
        target_update_interval=2000,    # Frequência para atualizar a rede alvo
        exploration_fraction=0.3,       # Determina o declínio do epsilon
        exploration_initial_eps=1.0,    # Começa com exploração no máximo
        exploration_final_eps=0.05,     # Exploração mínima no final
        verbose=1,
        tensorboard_log="./logs/"
    )

Treinar o Agente

In [ ]:
def train_agent(model, total_timesteps = 200000) -> None:
    """Train a DQN agent."""

    print("Starting training...\n")

    model_file_name = "dqn_moutain_car_model"
    model.learn(total_timesteps=total_timesteps, log_interval=10)
    model.save(f"models/{model_file_name}.zip")

    print(f"Training complete. Model saved as 'models/{model_file_name}.zip'")

Executar o Agente

In [ ]:
def run_agent(env, model_file_name, n_episodes: int=1):
    """Run the trained agent for multiple demo episodes."""

    print(f"Starting demo mode for {n_episodes} episodes...\n")

    videos_dir = "videos"
    env = RecordVideo(
        env,
        video_folder=videos_dir,
        episode_trigger=lambda e: True
    )
    model = DQN.load(f"models/{model_file_name}", env=env, device=device)
    total_rewards = []

    for episode in range(n_episodes):
        obs, _ = env.reset()
        done = False
        episode_reward = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward  # type: ignore (Pylance)

        total_rewards.append(episode_reward)
        print(f"Episode {episode + 1} | Reward: {episode_reward:.2f}")

    mean_reward = sum(total_rewards) / len(total_rewards)
    env.close()
    print(f"\n Demo finished | Mean reward over {n_episodes} episodes: {mean_reward:.2f}")

    video_files = [
        f for f in os.listdir(videos_dir)
        if f.endswith(".mp4")
    ]

    for video_file in video_files:
        display(Video(
            os.path.join(videos_dir, video_file),
            embed=True
        ))

## Treinar Modelo

In [ ]:
logger = configure(
    folder="./logs/",
    format_strings=["stdout", "tensorboard"]
)
model = create_model(env, POLICY, device)
train_agent(model, total_timesteps=800000)

## Executar Modelo

In [ ]:
run_agent(
    env,
    model_file_name="dqn_moutain_car_model"
)

## Exercícios

### Sobre o Ambiente

#### 1 - Qual ambiente você escolheu e por quê?

_O ambiente escolhido foi o **MountainCar-v0**._

_A escolha foi motivada principalmente por ser o ambiente mais simples entre as sugestões apresentadas, o que facilita entender o comportamento do algoritmo sem precisar lidar com uma complexidade muito grande logo de início._

_Além disso, o MountainCar é um problema clássico de controle bastante conhecido na literatura de aprendizado por reforço, o que torna mais fácil encontrar referências e comparar resultados._

---

#### 2 - Qual é o objetivo do agente nesse ambiente?

_O objetivo do agente é conduzir um carro até o topo de uma montanha, onde fica a bandeira._

_O problema é que o motor do carro não é forte o suficiente para subir a rampa diretamente, então o agente precisa aprender a balançar o carro para frente e para trás, ganhando impulso, até conseguir chegar ao topo._

_O episódio termina quando o carro alcança a posição da bandeira ou quando o limite de 200 passos é atingido._

---

#### 3 - Descreva as possíveis ações que o agente pode tomar, quais informações compõem os estados/observações, e como são definidas as recompensas?

_O agente tem 3 ações discretas disponíveis:_
- _0: Empurrar para a esquerda_
- _1: Não fazer nada (neutro)_
- _2: Empurrar para a direita_

_O estado é composto por 2 valores contínuos:_
- _**Posição** do carro (variando de `-1.2` a `0.6`)_
- _**Velocidade** do carro (variando de `-0.07` a `0.07`)_

_O agente recebe `-1` a cada passo de tempo, isso independentemente da ação tomada. O que incentiva o agente a chegar ao objetivo o mais rápido possível, já que quanto mais tempo levar, maior será a penalidade acumulada._

_Como o limite é de 200 passos, a recompensa mínima por episódio é `-200`, que é o que acontece quando o agente não consegue alcançar a bandeira._

---

### Sobre o Modelo Treinado

#### Com base no seu experimento, escreva um pequeno parágrafo detalhando a sua experiência, procurando responder às seguintes perguntas:

1. O algoritmo conseguiu encontrar uma estratégia válida para solucionar o problema do ambiente escolhido?

2. Se sim, qual configuração de parâmetros foi a mais bem sucedida? Quantos passos de treino levou para que o agente encontrasse essa solução? Caso contrário, por que você acha que não foi possível encontrar uma solução para esse ambiente?

3. Quais comportamentos emergiram durante o treinamento?

_O algoritmo conseguiu encontrar uma estratégia válida. Depois de algumas tentativas ajustando hiperparâmetros, a configuração que funcionou melhor foi:_
- `learning_rate=1e-4`
- `gamma=0.99`
- `buffer_size=50000`
- `exploration_fraction=0.3`
- `800.000 passos de treino total`

_Essa combinação foi importante porque o MountainCar tem um problema clássico com recompensas dispersas. O agente recebe `-1` a cada passo independentemente do que faz, então no começo do treino tudo parece igualmente ruim para ele. A `exploration_fraction=0.3` ajudou bastante nisso, dando ao agente uma janela longa de exploração para descobrir que balançar o carro para frente e para trás é o certo a se fazer._

_Sobre os comportamentos, no início do treino o agente ficava parado ou se movia de forma aleatória, esgotando os passos toda vez. Com o tempo, ele começou a desenvolver a estratégia de ganhar momentum — indo para a esquerda até a encosta oposta e aproveitando o impulso para subir a rampa da direita. Esse padrão de oscilação emergiu de forma gradual e foi claramente visível nos episódios gravados em vídeo. Não foi necessário nenhuma alteração no ambiente ou na estrutura da rede; o simples fato de dar passos suficientes de treino e manter a exploração ativa por tempo suficiente foi o que fez a diferença._